### ENTREGA

In [2]:
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
import time

In [20]:
# --------------------------------------------------
# RUTAS DE LOS ACCESORIOS
# --------------------------------------------------

carpeta = Path.cwd() / "accesorios"

ruta_sombrero = carpeta / "sombrero.png"
ruta_puro = carpeta / "puro.png"
ruta_bandana = carpeta / "bandadna.png"

# --------------------------------------------------
# CARGA DE IMAGENES
# --------------------------------------------------

def cargar_accesorio(ruta):
    imagen = cv2.imread(str(ruta), cv2.IMREAD_UNCHANGED)

    if imagen is None:
        print(f"No se ha encontrado: {ruta.name}")
        return None

    # Si la imagen no tiene transparencia, añadimos un canal alfa.
    if len(imagen.shape) == 3 and imagen.shape[2] == 3:
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2BGRA)

    return imagen


sombrero = cargar_accesorio(ruta_sombrero)
puro = cargar_accesorio(ruta_puro)
bandana = cargar_accesorio(ruta_bandana)
# Recortar el espacio transparente del puro.
if puro is not None:

    mascara = puro[:, :, 3]
    puntos = cv2.findNonZero(mascara)

    x, y, w, h = cv2.boundingRect(puntos)

    puro = puro[y:y+h, x:x+w]

# --------------------------------------------------
# SUPERPOSICION DE IMAGENES
# --------------------------------------------------

def superponer(imagen, accesorio, centro, ancho, angulo=0):

    if accesorio is None:
        return imagen

    alto_original, ancho_original = accesorio.shape[:2]

    # Mantener la proporcion original del accesorio.
    alto = int(alto_original * ancho / ancho_original)

    if ancho <= 0 or alto <= 0:
        return imagen

    accesorio = cv2.resize(accesorio, (ancho, alto))

    # Girar el accesorio para adaptarlo a la inclinacion de la cabeza.
    matriz = cv2.getRotationMatrix2D(
    (ancho // 2, alto // 2), -angulo, 1
    )

    accesorio = cv2.warpAffine(
        accesorio,
        matriz,
        (ancho, alto),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    alto_img, ancho_img = imagen.shape[:2]

    # Coordenadas de la esquina superior izquierda.
    x = int(centro[0] - ancho / 2)
    y = int(centro[1] - alto / 2)

    # Recortar si el accesorio sale de los limites de la imagen.
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(ancho_img, x + ancho)
    y2 = min(alto_img, y + alto)

    if x1 >= x2 or y1 >= y2:
        return imagen

    # Recortar tambien la imagen del accesorio.
    accesorio_recortado = accesorio[
        y1 - y:y2 - y,
        x1 - x:x2 - x
    ]

    # Separar los canales de color y transparencia.
    zona = imagen[y1:y2, x1:x2]

    alfa = accesorio_recortado[:, :, 3] / 255.0
    alfa = alfa[:, :, np.newaxis]

    colores = accesorio_recortado[:, :, :3]

    # Mezclar el accesorio con la imagen original.
    imagen[y1:y2, x1:x2] = (
        colores * alfa + zona * (1 - alfa)
    ).astype(np.uint8)

    return imagen

# --------------------------------------------------
# FILTRO DEL OESTE
# --------------------------------------------------

def filtro_oeste(imagen):

    # Convertir a float para realizar operaciones.
    resultado = imagen.astype(np.float32)

    # ------------------------------------------
    # TONO SEPIA
    # ------------------------------------------

    matriz_sepia = np.array([
        [0.272, 0.534, 0.131],
        [0.349, 0.686, 0.168],
        [0.393, 0.769, 0.189]
    ])

    resultado = cv2.transform(resultado, matriz_sepia)

    resultado = np.clip(resultado, 0, 255)

    # ------------------------------------------
    # CONTRASTE Y BRILLO
    # ------------------------------------------

    resultado = cv2.convertScaleAbs(
        resultado,
        alpha=1.15,
        beta=5
    )

    # ------------------------------------------
    # VIÑETEADO
    # ------------------------------------------

    alto, ancho = imagen.shape[:2]

    kernel_x = cv2.getGaussianKernel(ancho, ancho * 0.5)
    kernel_y = cv2.getGaussianKernel(alto, alto * 0.5)

    mascara = kernel_y @ kernel_x.T

    mascara = mascara / mascara.max()

    # Oscurecer ligeramente las esquinas.
    mascara = 0.55 + 0.45 * mascara

    resultado = (
        resultado.astype(np.float32) *
        mascara[:, :, np.newaxis]
    )

    # ------------------------------------------
    # GRANO CINEMATOGRAFICO
    # ------------------------------------------

    ruido = np.random.normal(
        0, 3, resultado.shape
    ).astype(np.float32)

    resultado = resultado + ruido

    return np.clip(resultado, 0, 255).astype(np.uint8)


# --------------------------------------------------
# DETECCION FACIAL
# --------------------------------------------------

modelo = Path.cwd() / "face_landmarker.task"

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

opciones = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(modelo)),
    running_mode=RunningMode.VIDEO,
    num_faces=1
)

face_landmarker = FaceLandmarker.create_from_options(opciones)


# --------------------------------------------------
# DETECCION DE MANOS
# --------------------------------------------------

modelo_manos = Path.cwd() / "hand_landmarker.task"

HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# --------------------------------------------------
# DETECCION DEL CUERPO
# --------------------------------------------------

modelo_pose = Path.cwd() / "pose_landmarker_lite.task"

PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions

opciones_pose = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(modelo_pose)
    ),
    running_mode=RunningMode.VIDEO,
    num_poses=1
)

pose_landmarker = PoseLandmarker.create_from_options(
    opciones_pose
)

opciones_manos = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(modelo_manos)
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=2
)

hand_landmarker = HandLandmarker.create_from_options(
    opciones_manos
)


# --------------------------------------------------
# CAPTURA DE VIDEO
# --------------------------------------------------

camara = cv2.VideoCapture(0)

if not camara.isOpened():
    print("No se ha podido acceder a la camara.")
    exit()

ultimo_timestamp = -1

# --------------------------------------------------
# ESTADO DEL PURO
# --------------------------------------------------

estado_puro = "boca"

tiempo_cerca = None

mano_alejada = False

tiempo_recogida = 0.9
tiempo_devolucion = 0.3

while True:

    correcto, frame = camara.read()

    if not correcto:
        break

    # Reflejar la imagen como si fuera un espejo.
    frame = cv2.flip(frame, 1)

    alto, ancho = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convertir la imagen al formato de MediaPipe.
    mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb
    )

# Obtener una marca de tiempo creciente.
    timestamp_ms = max(
    int(time.monotonic() * 1000),
    ultimo_timestamp + 1
    )
    ultimo_timestamp = timestamp_ms

# Detectar los puntos faciales.
    resultados = face_landmarker.detect_for_video(
    mp_image,
    timestamp_ms
    )

    # Detectar las manos.
    resultados_manos = hand_landmarker.detect_for_video(
    mp_image,
    timestamp_ms
    )

    # Detectar los puntos del cuerpo.
    resultados_pose = pose_landmarker.detect_for_video(
    mp_image,
    timestamp_ms
    )

    if resultados.face_landmarks:

        cara = resultados.face_landmarks[0]

        # Obtener las coordenadas de un punto facial.
        def punto(indice):
            landmark = cara[indice]

            x = int(landmark.x * ancho)
            y = int(landmark.y * alto)

            return (x, y)

        # Puntos principales de la cara.
        frente = punto(6)
        barbilla = punto(152)

        lado_izquierdo = punto(234)
        lado_derecho = punto(454)

        comisura_izquierda = punto(88)
        comisura_derecha = punto(291)

        boca_superior = punto(13)

        # Anchura aproximada de la cara.
        ancho_cara = int(
            np.linalg.norm(
                np.array(lado_derecho) -
                np.array(lado_izquierdo)
            )
        )

        # Inclinacion de la cabeza.
        vector_cara = (
            np.array(lado_derecho) -
            np.array(lado_izquierdo)
        )

        angulo = np.degrees(
            np.arctan2(vector_cara[1], vector_cara[0])
        )

        # ------------------------------------------
        # SOMBRERO
        # ------------------------------------------

        if sombrero is not None:

            ancho_sombrero = int(ancho_cara * 2.2)

            alto_sombrero = int(
                sombrero.shape[0] *
                ancho_sombrero /
                sombrero.shape[1]
            )

            centro_sombrero = (
                frente[0],
                frente[1] - int(alto_sombrero * 0.45)
            )

            frame = superponer(
                frame,
                sombrero,
                centro_sombrero,
                ancho_sombrero,
                angulo
            )



        # ------------------------------------------
        # BANDANA
        # ------------------------------------------

        if resultados_pose.pose_landmarks and bandana is not None:

            pose = resultados_pose.pose_landmarks[0]

            # Obtener la posicion de los hombros.
            hombro_izquierdo = (
                int(pose[11].x * ancho),
                int(pose[11].y * alto)
            )

            hombro_derecho = (
                int(pose[12].x * ancho),
                int(pose[12].y * alto)
            )

            # Calcular la distancia entre los hombros.
            distancia_hombros = int(
                np.linalg.norm(
                    np.array(hombro_derecho) -
                    np.array(hombro_izquierdo)
                )
            )

            # Centro entre ambos hombros.
            centro_hombros = (
                int((hombro_izquierdo[0] + hombro_derecho[0]) / 2),
                int((hombro_izquierdo[1] + hombro_derecho[1]) / 2)
            )

            # Desplazar la bandana ligeramente hacia abajo.
            centro_bandana = (
                centro_hombros[0],
                centro_hombros[1] + int(distancia_hombros * 0.06)
            )

            ancho_bandana = int(distancia_hombros * 1.10)

            # Calcular la inclinacion de los hombros.
            vector_hombros = (
                np.array(hombro_derecho) -
                np.array(hombro_izquierdo)
            )

            angulo_bandana = np.degrees(
                np.arctan2(
                    vector_hombros[1],
                    -vector_hombros[0]
                )
            )

            frame = superponer(
                frame,
                bandana,
                centro_bandana,
                ancho_bandana,
                -angulo_bandana
            )

        
# ------------------------------------------
# PURO E INTERACCION CON LA MANO
# ------------------------------------------

        if puro is not None:

            # Tamaño del puro respecto a la cara.
            ancho_puro = int(ancho_cara * 0.45)

            alto_puro = int(
                puro.shape[0] * ancho_puro / puro.shape[1]
            )

            # Escala del accesorio.
            escala = ancho_puro / puro.shape[1]

            # Punto de anclaje: extremo apagado del puro.
            ancla_x = int(ancho_puro * 0.96)
            ancla_y = int(alto_puro * 0.04)

            # La comisura será el punto donde se sujeta el puro.
            destino = comisura_derecha

            # Calcular el centro de la imagen para colocar
            # el extremo apagado exactamente en la comisura.
            centro_puro_boca = (
                destino[0] - ancla_x + ancho_puro,
                destino[1] - ancla_y + alto_puro // 2
            )

            # ------------------------------------------
            # DETECTAR POSICION DE LA MANO
            # ------------------------------------------

            punto_mano = None

            if resultados_manos.hand_landmarks:

                distancia_minima = float("inf")

                for mano in resultados_manos.hand_landmarks:

                    # Punta del pulgar e indice.
                    pulgar = mano[4]
                    indice = mano[8]

                    # Punto medio entre ambos dedos.
                    x_mano = int((pulgar.x + indice.x) / 2 * ancho)
                    y_mano = int((pulgar.y + indice.y) / 2 * alto)

                    candidato = (x_mano, y_mano)

                    # Buscar la mano mas cercana al puro.
                    distancia = np.linalg.norm(
                        np.array(candidato) -
                        np.array(centro_puro_boca)
                    )

                    if distancia < distancia_minima:
                        distancia_minima = distancia
                        punto_mano = candidato

            # Distancia entre la mano y el puro en la boca.
            if punto_mano is not None:

                distancia_boca = np.linalg.norm(
                    np.array(punto_mano) -
                    np.array(centro_puro_boca)
                )

            else:
                distancia_boca = float("inf")

            tiempo_actual = time.monotonic()

            umbral = int(ancho_cara * 0.20)

            # ------------------------------------------
            # ESTADO 1: PURO EN LA BOCA
            # ------------------------------------------

            if estado_puro == "boca":

                if distancia_boca < umbral:

                    if tiempo_cerca is None:
                        tiempo_cerca = tiempo_actual

                    elif tiempo_actual - tiempo_cerca >= tiempo_recogida:

                        estado_puro = "mano"
                        mano_alejada = False
                        tiempo_cerca = None

                else:
                    tiempo_cerca = None

                # Mostrar el puro en su posicion original.
                if estado_puro == "boca":

                    frame = superponer(
                        frame,
                        puro,
                        centro_puro_boca,
                        ancho_puro,
                        0
                    )

            # ------------------------------------------
            # ESTADO 2: PURO EN LA MANO
            # ------------------------------------------

            elif estado_puro == "mano":

                if punto_mano is not None:

                    # Esperar a que la mano se aleje.
                    if distancia_boca > umbral * 1.8:
                        mano_alejada = True

                    # Comprobar si vuelve a acercarse a la boca.
                    if mano_alejada and distancia_boca < umbral:

                        if tiempo_cerca is None:
                            tiempo_cerca = tiempo_actual

                        elif tiempo_actual - tiempo_cerca >= tiempo_devolucion:

                            estado_puro = "boca"
                            mano_alejada = False
                            tiempo_cerca = None

                    else:
                        tiempo_cerca = None

                    # Mostrar el puro en la mano.
                    if estado_puro == "mano":

                        centro_puro_mano = (
                            punto_mano[0] + int(ancho_puro * 0.20),
                            punto_mano[1]
                        )

                        frame = superponer(
                            frame,
                            puro,
                            centro_puro_mano,
                            ancho_puro,
                            0
                        )

                else:
                    tiempo_cerca = None


    # Aplicar el filtro del Oeste.
    frame = filtro_oeste(frame)

    # Mostrar resultado.
    cv2.imshow("Filtro de vaquero", frame)

    # Salir pulsando ESC.
    if cv2.waitKey(1) & 0xFF == 27:
        break


# --------------------------------------------------
# LIBERAR RECURSOS
# --------------------------------------------------

camara.release()
face_landmarker.close()
pose_landmarker.close()
cv2.destroyAllWindows()


In [18]:
# --------------------------------------------------
# DETECCION DE MANOS
# --------------------------------------------------

modelo_manos = Path.cwd() / "hand_landmarker.task"

BaseOptions = mp.tasks.BaseOptions
RunningMode = mp.tasks.vision.RunningMode

HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

opciones_manos = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(modelo_manos)
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=2
)

hand_landmarker = HandLandmarker.create_from_options(
    opciones_manos
)


# --------------------------------------------------
# FUNCION PARA DIBUJAR LA EMPUÑADURA
# --------------------------------------------------

def dibujar_empunadura(imagen, centro, direccion, tamano):

    # Comprobar que tenemos una dirección válida.
    magnitud = np.linalg.norm(direccion)

    if magnitud < 0.05:

        direccion = np.array([
            np.cos(np.radians(ultimo_angulo)),
            np.sin(np.radians(ultimo_angulo))
        ])

    # Vector perpendicular para calcular el grosor.
    perpendicular = np.array([
        -direccion[1],
        direccion[0]
    ])

    magnitud_perp = np.linalg.norm(perpendicular)

    if magnitud_perp > 0:
        perpendicular = perpendicular / magnitud_perp

    # Función auxiliar para calcular puntos sobre la empuñadura.
    def punto(posicion, desplazamiento=0):

        return (
            int(
                centro[0] +
                direccion[0] * tamano * posicion +
                perpendicular[0] * tamano * desplazamiento
            ),
            int(
                centro[1] +
                direccion[1] * tamano * posicion +
                perpendicular[1] * tamano * desplazamiento
            )
        )

    # Extremos de la empuñadura.
    parte_trasera = punto(-1.5)
    parte_delantera = punto(0.9)

    # Mango exterior.
    cv2.line(
        imagen,
        parte_trasera,
        parte_delantera,
        (25, 25, 30),
        int(tamano * 0.65),
        cv2.LINE_AA
    )

    # Cuerpo metálico interior.
    cv2.line(
        imagen,
        punto(-1.4),
        punto(0.75),
        (130, 135, 145),
        int(tamano * 0.42),
        cv2.LINE_AA
    )

    # Zona de agarre.
    cv2.line(
        imagen,
        punto(-1.15),
        punto(-0.15),
        (45, 45, 50),
        int(tamano * 0.48),
        cv2.LINE_AA
    )

    # Detalles de la zona de agarre.
    for posicion in [-1.0, -0.7, -0.4]:

        cv2.line(
            imagen,
            punto(posicion, -0.22),
            punto(posicion, 0.22),
            (105, 105, 115),
            max(1, int(tamano * 0.045)),
            cv2.LINE_AA
        )

    # Anillo metálico trasero.
    cv2.line(
        imagen,
        punto(-1.45),
        punto(-1.35),
        (190, 195, 205),
        int(tamano * 0.72),
        cv2.LINE_AA
    )

    # Anillo metálico delantero.
    cv2.line(
        imagen,
        punto(0.65),
        punto(0.85),
        (190, 195, 205),
        int(tamano * 0.75),
        cv2.LINE_AA
    )

    # Emisor de la espada.
    cv2.line(
        imagen,
        punto(0.85),
        punto(1.0),
        (75, 80, 90),
        int(tamano * 0.95),
        cv2.LINE_AA
    )

    # Pequeño detalle luminoso del emisor.
    cv2.line(
        imagen,
        punto(0.95),
        punto(1.0),
        (180, 200, 220),
        max(1, int(tamano * 0.18)),
        cv2.LINE_AA
    )

    # Botón de activación.
    boton = punto(-0.1, -0.28)

    cv2.circle(
        imagen,
        boton,
        max(2, int(tamano * 0.10)),
        (30, 40, 210),
        -1,
        cv2.LINE_AA
    )

    return imagen


# --------------------------------------------------
# FUNCION PARA SUPERPONER LA ESPADA
# --------------------------------------------------

def superponer_espada(imagen, accesorio, centro, ancho, angulo):

    if accesorio is None:
        return imagen

    alto_original, ancho_original = accesorio.shape[:2]

    alto = int(alto_original * ancho / ancho_original)

    accesorio = cv2.resize(accesorio, (ancho, alto))

    # Rotar la imagen conservando todo su contenido.
    matriz = cv2.getRotationMatrix2D(
        (ancho // 2, alto // 2),
        -angulo,
        1
    )

    coseno = abs(matriz[0, 0])
    seno = abs(matriz[0, 1])

    nuevo_ancho = int(alto * seno + ancho * coseno)
    nuevo_alto = int(alto * coseno + ancho * seno)

    matriz[0, 2] += nuevo_ancho / 2 - ancho / 2
    matriz[1, 2] += nuevo_alto / 2 - alto / 2

    accesorio = cv2.warpAffine(
        accesorio,
        matriz,
        (nuevo_ancho, nuevo_alto),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    alto_img, ancho_img = imagen.shape[:2]

    x = int(centro[0] - nuevo_ancho / 2)
    y = int(centro[1] - nuevo_alto / 2)

    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(ancho_img, x + nuevo_ancho)
    y2 = min(alto_img, y + nuevo_alto)

    if x1 >= x2 or y1 >= y2:
        return imagen

    accesorio_recortado = accesorio[
        y1-y:y2-y,
        x1-x:x2-x
    ]

    zona = imagen[y1:y2, x1:x2]

    alfa = accesorio_recortado[:, :, 3] / 255.0
    alfa = alfa[:, :, np.newaxis]

    colores = accesorio_recortado[:, :, :3]

    imagen[y1:y2, x1:x2] = (
        colores * alfa + zona * (1-alfa)
    ).astype(np.uint8)

    return imagen


# --------------------------------------------------
# FUNCION PARA DIBUJAR LA HOJA LASER
# --------------------------------------------------

def dibujar_hoja(imagen, inicio, fin, color):

    # Crear una capa para el resplandor.
    brillo = np.zeros_like(imagen)

    # Dibujar la parte exterior de la hoja.
    cv2.line(
        brillo,
        inicio,
        fin,
        color,
        25
    )

    # Difuminar para conseguir el efecto luminoso.
    brillo = cv2.GaussianBlur(
        brillo,
        (31, 31),
        0
    )

    imagen = cv2.addWeighted(
        imagen,
        1,
        brillo,
        0.8,
        0
    )

    # Dibujar el núcleo de la hoja.
    cv2.line(
        imagen,
        inicio,
        fin,
        color,
        10
    )

    # Núcleo más claro.
    cv2.line(
        imagen,
        inicio,
        fin,
        (255, 255, 255),
        3
    )

    return imagen


# --------------------------------------------------
# VARIABLES DE CONTROL
# --------------------------------------------------

ultima_posicion_mano = None

ultimo_timestamp = -1
ultimo_angulo = 0

# Color provisional de la hoja (BGR).
color_hoja = (255, 180, 50)

# Ajuste de orientación de la imagen.
angulo_offset = 0


# --------------------------------------------------
# CAPTURA DE VIDEO
# --------------------------------------------------

camara = cv2.VideoCapture(0)

if not camara.isOpened():
    print("No se ha podido acceder a la cámara.")

else:

    while True:

        correcto, frame = camara.read()

        if not correcto:
            break

        # Reflejar la imagen como si fuera un espejo.
        frame = cv2.flip(frame, 1)

        alto, ancho = frame.shape[:2]

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = max(
            int(time.monotonic() * 1000),
            ultimo_timestamp + 1
        )

        ultimo_timestamp = timestamp_ms

        # Detectar las manos.
        resultados = hand_landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        manos = resultados.hand_landmarks

        # --------------------------------------------------
        # CALCULAR POSICION DE LAS MANOS
        # --------------------------------------------------

        posiciones = []
        tamanos = []

        for mano in manos:

            # Muñeca y nudillo del dedo corazón.
            muneca = mano[0]
            nudillo = mano[9]

            punto_muneca = (
                int(muneca.x * ancho),
                int(muneca.y * alto)
            )

            punto_nudillo = (
                int(nudillo.x * ancho),
                int(nudillo.y * alto)
            )

            # Centro aproximado de la palma.
            centro = punto_nudillo

            # Tamaño aproximado de la mano.
            tamano = np.linalg.norm(
                np.array(punto_nudillo) -
                np.array(punto_muneca)
            )

            posiciones.append(centro)
            tamanos.append(tamano)

        # --------------------------------------------------
        # SELECCIONAR LA MANO QUE SUJETA LA ESPADA
        # --------------------------------------------------

        punto_espada = None
        tamano_mano = None
        angulo = ultimo_angulo

        if len(posiciones) > 0:

            # En el primer frame utilizamos la primera mano.
            # Después intentamos mantener la misma mediante
            # la proximidad a su última posición.
            if ultima_posicion_mano is None:

                indice_mano = 0

            else:

                distancias = [
                    np.linalg.norm(
                        np.array(posicion) -
                        np.array(ultima_posicion_mano)
                    )
                    for posicion in posiciones
                ]

                indice_mano = int(np.argmin(distancias))

            punto_espada = posiciones[indice_mano]
            tamano_mano = tamanos[indice_mano]

            # Guardar la posición para el siguiente frame.
            ultima_posicion_mano = punto_espada

            mano = manos[indice_mano]

            # Obtener los nudillos del índice y del meñique.
            indice = mano[5]
            menique = mano[17]

            # Calcular el vector tridimensional.
            dx = (indice.x - menique.x) * ancho
            dy = (indice.y - menique.y) * alto
            dz = (indice.z - menique.z) * ancho

            # Magnitud del vector 3D.
            magnitud = np.sqrt(dx**2 + dy**2 + dz**2)

            if magnitud > 0.00001:

                # Calcular la orientación 2D para la empuñadura.
                distancia_2d = np.sqrt(dx**2 + dy**2)

                if distancia_2d > 15:

                    angulo = np.degrees(
                        np.arctan2(dy, dx)
                    )

                    ultimo_angulo = angulo

                # Proyectar el vector 3D sobre la imagen.
                # No normalizamos esta proyección para conservar
                # el efecto de profundidad.
                direccion = np.array([
                    dx / magnitud,
                    dy / magnitud
                ])

            else:

                # Si el vector no es válido, conservar la última
                # orientación y utilizar una dirección 2D.
                direccion = np.array([
                    np.cos(np.radians(ultimo_angulo)),
                    np.sin(np.radians(ultimo_angulo))
                ])

        else:

            ultima_posicion_mano = None

        
        # --------------------------------------------------
        # DIBUJAR LA ESPADA
        # --------------------------------------------------

        if punto_espada is not None:

            # Dibujar la empuñadura utilizando la misma dirección
            # tridimensional proyectada que utiliza la hoja.
            frame = dibujar_empunadura(
                frame,
                punto_espada,
                direccion,
                tamano_mano
            )

            # La hoja comienza cerca del emisor.
            inicio = (
                int(punto_espada[0] + direccion[0] * tamano_mano * 0.8),
                int(punto_espada[1] + direccion[1] * tamano_mano * 0.8)
            )

            # Longitud de la hoja.
            fin = (
                int(punto_espada[0] + direccion[0] * tamano_mano * 6),
                int(punto_espada[1] + direccion[1] * tamano_mano * 6)
            )

            frame = dibujar_hoja(
                frame,
                inicio,
                fin,
                color_hoja
            )


        # --------------------------------------------------
        # MOSTRAR RESULTADO
        # --------------------------------------------------

        cv2.imshow("Espada laser", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break


# --------------------------------------------------
# LIBERAR RECURSOS
# --------------------------------------------------

camara.release()
hand_landmarker.close()
cv2.destroyAllWindows()